In [1]:
import os
from bs4 import BeautifulSoup
from playwright.async_api import async_playwright, TimeoutError as PlaywrightTimeout
import time 

In [2]:
SEASONS = list(range(2021,2026))

In [3]:
DATA_DIR = "data"
STANDINGS_DIR = os.path.join(DATA_DIR,"standings")
SCORES_DIR = os.path.join(DATA_DIR,"scores")

In [4]:
from playwright.async_api import TimeoutError as PlaywrightTimeout

async def get_html(url, selector, sleep=5, retries=3):
    html = None
    for i in range(1, retries + 1):
        time.sleep(sleep)
        try:
            async with async_playwright() as p:
                browser = await p.firefox.launch(headless=True)
                page = await browser.new_page()
                await page.goto(url, timeout=60000)  # give it more time, e.g. 60s
                print(await page.title())
                html = await page.inner_html(selector)
        except (PlaywrightTimeout, TimeoutError) as e:
            print(f"Timeout error on {url}: {e}")
            continue
        else:
            break
        finally:
            await browser.close()
    return html

In [5]:
async def scrape_season(season):
    url = f"https://www.basketball-reference.com/leagues/NBA_{season}_games.html"
    html = await get_html(url, "#content .filter")
    
    soup = BeautifulSoup(html)
    links = soup.find_all("a")
    href = [l["href"]for l in links]
    standings_pages =  [f"https://basketball-reference.com{l}" for l in href]

    for url in standings_pages:
        save_path = os.path.join(STANDINGS_DIR, url.split("/")[-1])
        if os.path.exists(save_path):
            
            continue

        html = await get_html(url, "#all_schedule")

        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        with open(save_path, "w+") as f:
            f.write(html)

In [6]:
for season in SEASONS:
    await scrape_season(season)

2020-21 NBA Schedule | Basketball-Reference.com
2021-22 NBA Schedule | Basketball-Reference.com
Timeout error on https://www.basketball-reference.com/leagues/NBA_2023_games.html: Page.goto: Timeout 60000ms exceeded.
Call log:
  - navigating to "https://www.basketball-reference.com/leagues/NBA_2023_games.html", waiting until "load"

2022-23 NBA Schedule | Basketball-Reference.com
2023-24 NBA Schedule | Basketball-Reference.com
2024-25 NBA Schedule | Basketball-Reference.com


In [7]:
standings_files = os.listdir(STANDINGS_DIR)

In [8]:
async def scrape_game(standings_file):
    with open(standings_file,'r') as f:
        html = f.read()

    soup = BeautifulSoup(html)
    links = soup.find_all("a")
    hrefs = [l.get("href")for l in links]
    box_scores = [l for l in hrefs if l and "boxscore" in l and ".html" in l]
    box_scores = [f"https://www.basketball-reference.com{l}" for l in box_scores]

    for url in box_scores:
        save_path = os.path.join(SCORES_DIR,url.split("/")[-1])
        if os.path.exists(save_path):
            continue

        html = await get_html (url,"#content")
        if not html:
            continue
        with open (save_path,"w+")as f:
            f.write(html)

In [ ]:
    for f in standings_files:
        filepath = os.path.join(STANDINGS_DIR, f)

        await scrape_game(filepath) 